# Student allocation to projects

## Getting started
Create 2 csv files or amend them with actual data
- `student_preferences.csv` with 1 + number of preferences columns: student, preference1, preference2, ..., preferenceN
- `project_capacities.csv` with 3 columns: project, capacity, supervisor

Then run every cell, the output are two csv files
- `matching_student.csv` for a student focused view with 2 columns: student, project allocated, preference rank of allocation, list of preferences
- `matching_project.csv` for a project focused view with 4 columns: project, list of student allocated, capacity, supervisor

## Other functions
- `check_matching` checks the validity
- `shuffle_students` randomly moves the order of the student. The order of students will affect the allocations, the student at the top will get largest priority of their choice of projects. They will, by definition, get their first choice.

In [1]:
import pandas as pd
import numpy as np

# Define student and project classes
class Student:
    def __init__(self, id, preferences):
        self.id = id
        self.preferences = preferences
        self.matched_project = None
        self.rank_choice = 1 # 1 indexed for readability

class Project:
    def __init__(self, id, capacity):
        self.id = id
        self.capacity = capacity
        self.matched_students = []
        self.applicant_students = []

# Define functions to help algorithm
def assign_student_to_project(student, project):
    project.matched_students.append(student)
    student.matched_project = project

def remove_student_from_project(student, project):
    project.matched_students.remove(student)
    student.matched_project = None

# Function to check validity of matching
def check_matching(students, projects):
    # check no student is matched to more than one project
    for s in students:
        if s.matched_project is not None:
            for p in projects:
                if p.id != s.matched_project.id and p == s.matched_project:
                    print(f"Major Warning - Student {s.id} is matched to multiple projects")
                    return False
            
    # check no project is matched to more than its capacity
    for p in projects:
        if len(p.matched_students) > p.capacity:
            print(f"Major Warning - Project {p.id} is matched to more students than its capacity")
            return False
        
    # warning - check if projects were preferred by students
    for s in students:
        if s.matched_project is not None and s.matched_project.id not in s.preferences:
            print(f"Minor Warning - Student {s.id} is matched to a project they did not prefer")
    
    # warning - check any unassigned students
    if any(s.matched_project is None for s in students):
        print(f"Minor Warning - Some students are not matched to any project")
        
    # valid
    print("Matching is valid")
    return True

# Function to shuffle students
def shuffle_students(students):
    np.random.shuffle(students)

# rate matching
def rate_matching(students):
    score = 0
    for s in students:
        if s.matched_project is not None:
            score += 10 * s.rank_choice **-2
    return score


In [2]:
# Gale-Shapley Algorithm implementation
def gale_shapley(students, projects, n_prefs=None):
    # By default, use all preferences
    if n_prefs is None:
        n_prefs = len(students[0].preferences)

    # Initialize all students and projects as unmatched
    for student in students:
        student.matched_project = None
    for project in projects:
        project.matched_students = []

    # Create a dictionary to easily access projects by their id
    project_dict = {p.id: p for p in projects}

    # Continue until all students have proposed to all their preferences
    unconsidered_students = students.copy()
    round_count = 0

    while unconsidered_students and round_count < (len(students) * len(projects)):
        round_count += 1
        
        for student in unconsidered_students.copy():
            if student.rank_choice <= min(n_prefs, len(student.preferences)):
                # Student proposes to their preference
                project_id = student.preferences[student.rank_choice - 1]
                project = project_dict[project_id]
                
                if project.capacity == 0:
                    # Project is removed, move to next preference
                    student.rank_choice += 1
                elif len(project.matched_students) < project.capacity:
                    # Project has capacity, accept the student
                    assign_student_to_project(student, project)
                    unconsidered_students.remove(student)
                else:
                    # Project is at capacity, compare with current matches
                    # The student with the lowest rank choice (highest preference) is the preferred student
                    # e.g. studentA.rank_choice = 1, studentB.rank_choice = 2, studentA is preferred
                    worst_match = max(project.matched_students, key=lambda s: s.rank_choice)
                    if student.rank_choice < worst_match.rank_choice:
                        # New student is preferred, swap
                        remove_student_from_project(worst_match, project)
                        assign_student_to_project(student, project)
                        unconsidered_students.remove(student)
                        unconsidered_students.append(worst_match)
                    else:
                        # Student is rejected, move to next preference
                        student.rank_choice += 1
            else:
                # Student has proposed to all preferences, remove from unconsidered_students
                student.rank_choice = 0
                unconsidered_students.remove(student)

    return students, projects


In [3]:
# Example data - using csv

student_preferences = pd.read_csv("student_preferences.csv")
# N+1 columns: student, preference1, preference2, ..., preferenceN

project_capacities = pd.read_csv("project_capacities.csv")
# 3 columns: project, capacity, supervisor


students = []
projects = []

for index, row in student_preferences.iterrows():
    students.append(Student(row["student"], row[1:].tolist()))

for index, row in project_capacities.iterrows():
    projects.append(Project(row["project"], row["capacity"]))


In [4]:
# ######################
# ## Make change here ##
# ######################
# # Run the optimal seed search using `seed_search_par.py`. Set the optimal seed here.

# # set optimal seed here
# optimize_seed = 8001

# np.random.seed(optimize_seed)
# shuffle_students(students)

In [5]:
# Run the algorithm
# By default, we will consider all given preferences
N_PREFERENCES_TO_CONSIDER = student_preferences.shape[1] - 1

gale_shapley(students, projects, N_PREFERENCES_TO_CONSIDER)

# Print the results
print("==========")
print("Student Assignments:")
for student in students:
    print(f"Student {student.id} matched to Project {student.matched_project.id if student.matched_project is not None else 'None'} with rank choice {student.rank_choice}")

print("\nProject Assignments:")
for project in projects:
    print(f"Project {project.id} matched to students: {[s.id for s in project.matched_students]}")

Student Assignments:
Student 1 matched to Project Project1 with rank choice 1
Student 2 matched to Project Project10 with rank choice 1
Student 3 matched to Project Project4 with rank choice 1
Student 4 matched to Project Project12 with rank choice 1
Student 5 matched to Project Project11 with rank choice 1
Student 6 matched to Project Project3 with rank choice 1
Student 7 matched to Project Project21 with rank choice 2
Student 8 matched to Project Project14 with rank choice 1
Student 9 matched to Project Project6 with rank choice 1
Student 10 matched to Project Project19 with rank choice 1
Student 11 matched to Project Project20 with rank choice 2
Student 12 matched to Project Project5 with rank choice 1
Student 13 matched to Project Project5 with rank choice 1
Student 14 matched to Project Project20 with rank choice 5
Student 15 matched to Project Project7 with rank choice 1
Student 16 matched to Project None with rank choice 0
Student 17 matched to Project Project13 with rank choice

In [6]:
print(f'Matching score: {rate_matching(students)}')

Matching score: 147.9


# Check validity and export

In [7]:
# check validity of matching
check_matching(students, projects)

Minor Warning - Some students are not matched to any project
Matching is valid


True

In [8]:
# save as pandas
matching_student_df = pd.DataFrame(columns=["student", "project", "rank_choice", "preference"])
for i in range(len(students)):
    student = students[i]
    if student.matched_project is not None:
        matching_student_df.loc[i] = [student.id, student.matched_project.id, student.rank_choice, student.preferences]
    else: 
        matching_student_df.loc[i] = [student.id, None, 0, student.preferences]

matching_project_df = pd.DataFrame(columns=["project", "students", "capacity"])
for i in range(len(projects)):
    project = projects[i]
    matching_project_df.loc[i] = [project.id, [s.id for s in project.matched_students], project.capacity]

# join project_capacities
matching_project_df = matching_project_df.merge(project_capacities[["project","supervisor"]], left_on="project", right_on="project", how="left")

# # export to csv
matching_student_df.to_csv("matching_student.csv", index=False)
matching_project_df.to_csv("matching_project.csv", index=False)

## Evaluation of allocations

In [9]:
matching_student_df

,student,project,rank_choice,preference
0,1,Project1,1,"[Project1, Project20, Project3, Project21, Pro..."
1,2,Project10,1,"[Project10, Project1, Project4, Project3, Proj..."
2,3,Project4,1,"[Project4, Project3, Project19, Project21, Pro..."
3,4,Project12,1,"[Project12, Project21, Project13, Project20, P..."
4,5,Project11,1,"[Project11, Project1, Project14, Project1, Pro..."
5,6,Project3,1,"[Project3, Project22, Project13, Project3, Pro..."
6,7,Project21,2,"[Project12, Project21, Project4, Project21, Pr..."
7,8,Project14,1,"[Project14, Project17, Project2, Project1, Pro..."
8,9,Project6,1,"[Project6, Project8, Project10, Project22, Pro..."
9,10,Project19,1,"[Project19, Project20, Project1, Project21, Pr..."


In [10]:
# total matched students
f'Total matched students: {len(matching_student_df[~matching_student_df["project"].isnull()])}'

'Total matched students: 18'

In [11]:
f'Students without projects: {len(matching_student_df[matching_student_df["project"].isnull()])}'

'Students without projects: 2'

In [12]:
f'Unmatched projects: {len(matching_project_df[(matching_project_df["students"].apply(len) == 0) & (matching_project_df["capacity"]>0)])}'

'Unmatched projects: 11'

In [13]:
matching_student_df.groupby('rank_choice').student.count()

rank_choice
0     2
1    14
2     3
5     1
Name: student, dtype: int64

In [14]:
# empty projects
matching_project_df[(matching_project_df["students"].apply(len) == 0) & (matching_project_df["capacity"]>0)]

,project,students,capacity,supervisor
7,Project8,[],3,B
8,Project9,[],1,C
14,Project15,[],1,D
15,Project16,[],1,D
16,Project17,[],1,E
17,Project18,[],3,E
22,Project23,[],1,G
23,Project24,[],1,G
24,Project25,[],1,G
26,Project27,[],1,H


In [15]:
# projects with remaining capacity
matching_project_df[(matching_project_df["capacity"] - matching_project_df["students"].apply(len)) > 0]

,project,students,capacity,supervisor
6,Project7,[15],2,B
7,Project8,[],3,B
8,Project9,[],1,C
14,Project15,[],1,D
15,Project16,[],1,D
16,Project17,[],1,E
17,Project18,[],3,E
22,Project23,[],1,G
23,Project24,[],1,G
24,Project25,[],1,G
